In [23]:
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm
import statsmodels.api as sm
from statsmodels.stats.power import tt_ind_solve_power, tt_solve_power

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

# Простой частотный А/Б тест.

## Постановка задачи

![banner](https://storage.googleapis.com/kaggle-datasets-images/635/1204/126be74882028aac7241553cef0e27a7/dataset-original.jpg)

Покемоны - это маленькие существа, которые сражаются друг с другом на соревнованиях. Все покемоны имеют разные характеристики (сила атаки, защиты и т. д.) И относятся к одному или двум так называемым классам (вода, огонь и т. д.).
Профессор Оук является изобретателем Pokedex, портативного устройства, которое хранит информацию обо всех существующих покемонах. Как его ведущий специалист по данным, Вы только что получили от него запрос с просьбой осуществить аналитику данных на всех устройствах Pokedex. 

## Описание набора данных
Профессор Оук скопировал все содержимое памяти одного устройства Pokedex, в результате чего получился набор данных, с которым Вы будете работать в этой задаче. В этом файле каждая строка представляет характеристики одного покемона:

* `pid`: Numeric - ID покемона
* `HP`: Numeric - Очки здоровья
* `Attack`: Numeric - Сила обычной атаки
* `Defense`: Numeric - Сила обычной защиты
* `Sp. Atk`: Numeric - Сила специальной атаки
* `Sp. Def`: Numeric - Сила специальной защиты
* `Speed`: Numeric - Скорость движений
* `Legendary`: Boolean - «True», если покемон редкий
* `Class 1`: Categorical - Класс покемона
* `Class 2`: Categorical - Класс покемона

## Загрузка данных

In [3]:
pokemon_path = 'https://raw.githubusercontent.com/a-milenkin/datasets_for_t-tests/main/pokemon.csv'
pokemon = pd.read_csv(pokemon_path)  # Откроем датасет
pokemon.head()

# Обратите внимание, что у покемона может быть один или два класса. Если у покемона два класса, считается,
# что они имеют одинаковую значимость

,pid,Name,Class 1,Class 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Legendary
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,False
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,False
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,False
3,4,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,False
4,5,Charmander,Fire,NaN,39,52,43,60,50,65,False


<div class="alert alert-info">
<b>Задание:</b>
    
Профессор Оук подозревает, что покемоны в классе `grass` имеют более сильную обычную атаку, чем у покемонов в классе `rock`. Проверьте, прав ли он, и убедите его в своем выводе статистически.
    
    
Примечание: если есть покемоны, которые относятся к обоим классам, просто выбросьте их.
    
Вы можете предположить, что распределение обычных атак является нормальным для всех классов покемонов.

</div>

In [4]:
alpha = 0.05

# 1. Отбор нужных данных
grass_attack = pokemon[pokemon['Class 1'] == 'Grass']['Attack']
rock_attack = pokemon[pokemon['Class 1'] == 'Rock']['Attack']

### Проверка нормальности данных

In [5]:
# 2. Проверка нормальности
# Если данных мало - то тест Шапиро-Уилка
if grass_attack.shape[0] < 20000:
    print('Проверка нормальности - Тест Шапиро-Уилка')
    stat, p_value_g_sh = stats.shapiro(grass_attack)
    if p_value_g_sh < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Grass отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Grass распределены нормально.{bcolors.ENDC}")

    stat, p_value_r_sh = stats.shapiro(rock_attack)
    if p_value_r_sh < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Rock отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Rock распределены нормально.{bcolors.ENDC}")
else:
    # если данных много - то тест Колмогорова-Смирнова
    print('Проверка нормальности - Тест Колмогорова-Смирнова')
    stat, p_value_g_ks = stats.kstest(grass_attack, 'norm')
    if p_value_g_ks < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Grass отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Grass распределены нормально.{bcolors.ENDC}")

    stat, p_value_r_ks = stats.kstest(rock_attack, 'norm')
    if p_value_r_ks < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Rock отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Rock распределены нормально.{bcolors.ENDC}")


Проверка нормальности - Тест Шапиро-Уилка
Не отвергаем нулевую гипотезу: данные Grass распределены нормально.
Не отвергаем нулевую гипотезу: данные Rock распределены нормально.


#### Преобразование Бокса-Кокса
Если нормальное распределение данных нарушено, это повод задуматься, что не так в наших данных.

In [6]:
grass_attack_transformed, grass_attack_fitted_lambda = stats.boxcox(grass_attack)
print(f'Lambda for grass: {grass_attack_fitted_lambda}')
rock_attack_transformed, rock_attack_fitted_lambda = stats.boxcox(rock_attack)
print(f'Lambda for rock: {rock_attack_fitted_lambda}')

Lambda for grass: 0.6520438335127198
Lambda for rock: 0.47258378976233345


In [7]:
# 2. Проверка нормальности на преобразованных данных
# Если данных мало - то тест Шапиро-Уилка
if grass_attack.shape[0] < 20000:
    print('Проверка нормальности - Тест Шапиро-Уилка')
    stat, p_value_g_sh = stats.shapiro(grass_attack_transformed)
    if p_value_g_sh < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Grass отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Grass распределены нормально.{bcolors.ENDC}")

    stat, p_value_r_sh = stats.shapiro(rock_attack_transformed)
    if p_value_r_sh < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Rock отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Rock распределены нормально.{bcolors.ENDC}")
else:
    # если данных много - то тест Колмогорова-Смирнова
    print('Проверка нормальности - Тест Колмогорова-Смирнова')
    stat, p_value_g_ks = stats.kstest(grass_attack_transformed, 'norm')
    if p_value_g_ks < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Grass отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Grass распределены нормально.{bcolors.ENDC}")

    stat, p_value_r_ks = stats.kstest(rock_attack_transformed, 'norm')
    if p_value_r_ks < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: данные Rock отклоняются от нормального распределения.{bcolors.ENDC}")
    else:
        print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: данные Rock распределены нормально.{bcolors.ENDC}")

Проверка нормальности - Тест Шапиро-Уилка
Не отвергаем нулевую гипотезу: данные Grass распределены нормально.
Не отвергаем нулевую гипотезу: данные Rock распределены нормально.


<div class="alert alert-warning">
<b>Нарушение нормальности распределений показателя</b>

Если даже после преобразования Бокса-Кокса мы видим **нарушение нормальности** распределения нашей величины - то **нельзя использовать t-тест Стьюдента и мы должны использовать тест Мана-Уитни**.
</div>

### Проверка дисперсии

#### Тест Левена

In [9]:
# Применение теста Левена
stat, p_value = stats.levene(grass_attack_transformed, rock_attack_transformed)

# Вывод результатов
print(f"Статистика Левена: {np.round(stat, 4)}")
print(f"p-значение: {np.round(p_value, 4)}")

# Интерпретация результата
alpha = 0.05
if p_value < alpha:
    print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: дисперсии не равны.{bcolors.ENDC}")
else:
    print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: дисперсии равны.{bcolors.ENDC}")


print(grass_attack_transformed.std())
print(rock_attack_transformed.std())

Статистика Левена: 16.0412
p-значение: 0.0001
Отвергаем нулевую гипотезу: дисперсии не равны.
5.739482583109376
3.2684230301768102


<div class="alert alert-warning">
<b>Нарушение равенства дисперсий</b>

Если дисперсии не равны, то мы должны использовать не клссический тест Стьюдента, а тест Уэлча, который не требователен к равенству дисперсий.
</div>

### t-тест Стьюдента

Классический t-тест

In [14]:
t_stat, p_value = stats.ttest_ind(grass_attack_transformed, rock_attack_transformed, equal_var=True)  # equal_var=True - классический тест Стьюдента
if p_value < alpha:
        print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: средние в выборках различаются.{bcolors.ENDC}")
else:
    print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: средние равны.{bcolors.ENDC}")

Отвергаем нулевую гипотезу: средние в выборках различаются.


### Тест Уэлча

In [13]:
t_stat, p_value = stats.ttest_ind(grass_attack, rock_attack, equal_var=False)  # Лучше всегда ставить equal_var=False (тест Уэлча)
if p_value < alpha:
    print(f"{bcolors.FAIL}Отвергаем нулевую гипотезу: средние в выборках различаются.{bcolors.ENDC}")
else:
    print(f"{bcolors.OKGREEN}Не отвергаем нулевую гипотезу: средние равны.{bcolors.ENDC}")

Отвергаем нулевую гипотезу: средние в выборках различаются.


### Тест Мана-Уитни

непараметрический тест, который применяется для проверки гипотезы о том, что два независимых выборки происходят из одного и того же распределения.

#### Когда используется:
1.	Сравнение двух независимых выборок.
2.	Когда данные не соответствуют нормальному распределению.
3.	Когда переменная измеряется по порядковой шкале (ранги).
4.	Когда в выборках много выбросов или маленький размер выборок.

#### Формулировка гипотез:
- H0 (нулевая гипотеза): Распределения двух выборок одинаковы.
- H1 (альтернативная гипотеза): Распределения различаются.

In [16]:
# Сравнение групп
stat, p = stats.mannwhitneyu(grass_attack, rock_attack) # U-тест Манна-Уитни
print('Statistics=%.3f, p=%.3f' % (stat, p))


# Интерпретируем
alpha = 0.05   # Уровень значимости
if p < alpha:
    print(f'{bcolors.FAIL}Одинаковые распределения (не отвергаем H0){bcolors.ENDC}')
else:
    print(f'{bcolors.OKGREEN}Разные распределения (отвергаем H0){bcolors.ENDC}')

Statistics=1059.000, p=0.005
Одинаковые распределения (не отвергаем H0)


## Расчет размера экспериментальной выборки

In [31]:
# 1. Считаем средние и стандартные отклонения в каждой группе
mean_grass = np.mean(grass_attack)
mean_rock = np.mean(rock_attack)
std_grass = np.std(grass_attack, ddof=1)
std_rock = np.std(rock_attack, ddof=1)

# 2. Считаем "pooled" стандартное отклонение (предполагаем равные дисперсии)
n_grass = len(grass_attack)
n_rock = len(rock_attack)
pooled_std = np.sqrt(
    ((n_grass - 1)*std_grass**2 + (n_rock - 1)*std_rock**2) / 
    (n_grass + n_rock - 2)
)

# 3. Вычисляем эффект размера (Cohen's d)
effect_size = abs(mean_grass - mean_rock) / pooled_std


analysis = sm.stats.TTestIndPower()
stat_size = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8)

print(f"Необходимое число наблюдений: {stat_size:.2f}")

Необходимое число наблюдений: 36.60
